In [66]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [67]:
final_data = Path.cwd().parent.parent.joinpath('processed_data', 'final_dataset.csv')
candidates_ranked_data = Path.cwd().parent.parent.joinpath('processed_data', 'candidates_ranked.csv')

In [68]:
df = pd.read_csv(candidates_ranked_data)
df.head(10)

,geo_id,county_name,population,median_household_income,housing_units,total_energy_consumption_mwh,data_centers_count,sfha_area,pct_sfha,lake_count,...,max_voltage,average_voltage,protected_count,total_protected_area_m,pct_protected,county_area_km2,population_density,has_plant,mcda_score,rank
0,40075,Kiowa County,8181.0,42679.0,4700.0,157935.0,0.0,2.112674e+08,0.079146,10,...,138.0,110.400000,2.0,4.108833e+07,0.015393,2669.336760,3.064806,0,0.529671,1880.0
1,46079,Lake County,10993.0,74884.0,5714.0,170505.0,0.0,1.890356e+08,0.126898,48,...,69.0,69.000000,156.0,1.136746e+08,0.076309,1489.662955,7.379522,0,0.591694,1349.0
2,37033,Caswell County,22563.0,56999.0,10493.0,295896.0,0.0,6.618573e+07,0.059609,4,...,230.0,230.000000,16.0,1.409890e+07,0.012698,1110.336635,20.320864,0,0.637095,799.0
3,48377,Presidio County,5433.0,29012.0,3396.0,106309.0,0.0,0.000000e+00,0.000000,0,...,69.0,69.000000,16.0,3.486332e+08,0.034909,9986.831983,0.544016,0,0.585314,1404.0
4,39057,Greene County,174322.0,81243.0,71471.0,2239244.0,0.0,1.002618e+08,0.092999,8,...,345.0,112.909091,58.0,1.130754e+07,0.010488,1078.100699,161.693616,0,0.596162,1284.0
5,35028,Los Alamos County,19407.0,135801.0,8631.0,282099.0,0.0,1.440459e+06,0.005093,0,...,115.0,115.000000,14.0,2.715960e+07,0.096026,282.834897,68.616003,0,0.530080,1877.0
6,20041,Dickinson County,18637.0,62971.0,8785.0,264300.0,0.0,3.515537e+08,0.159324,6,...,345.0,156.818182,7.0,7.963168e+06,0.003609,2206.527868,8.446302,0,0.579372,1469.0
7,48427,Starr County,66319.0,35979.0,22828.0,1104219.0,0.0,3.567815e+08,0.112075,1,...,345.0,142.000000,39.0,5.840314e+07,0.018346,3183.406363,20.832716,0,0.643165,743.0
8,40129,Roger Mills County,3259.0,57574.0,1844.0,64468.0,0.0,2.289701e+08,0.077115,10,...,230.0,174.800000,2.0,7.284311e+04,0.000025,2969.213529,1.097597,0,0.612842,1072.0
9,55103,Richland County,17036.0,61985.0,8507.0,204452.0,0.0,1.160961e+08,0.076065,2,...,161.0,74.750000,96.0,4.811957e+07,0.031527,1526.273724,11.161825,0,0.653788,644.0


#### Define Predicting ()

In [69]:
df.columns

Index(['geo_id', 'county_name', 'population', 'median_household_income',
       'housing_units', 'total_energy_consumption_mwh', 'data_centers_count',
       'sfha_area', 'pct_sfha', 'lake_count', 'total_lake_area', 'avg_vol',
       'avg_depth', 'avg_discharge', 'dist_to_lakes_km', 'wetland_count',
       'distance_to_rivers_km', 'rivers_count', 'total_rivers_mile',
       'military_count', 'total_military_area_m', 'pct_military',
       'plant_count', 'pga_max', 'distance_to_lines_km',
       'transmission_lines_count', 'max_voltage', 'average_voltage',
       'protected_count', 'total_protected_area_m', 'pct_protected',
       'county_area_km2', 'population_density', 'has_plant', 'mcda_score',
       'rank'],
      dtype='object')

#### Define Features
**Identify correlation between variables using correlation matrix**


In [70]:
features = [
    "pga_max",
    "pct_sfha",
    "population_density",
    "dist_to_lakes_km",
    "avg_vol",
    "total_lake_area",
    "distance_to_lines_km",
    "max_voltage",
    "total_energy_consumption_mwh",
    "data_centers_count",
    "pct_military",
    "pct_protected"
]

#### Logtistic Model

In [71]:
#define
X = df[features]
y = df['has_plant']

In [72]:
X.isna().sum()

pga_max                          0
pct_sfha                         0
population_density              11
dist_to_lakes_km                 0
avg_vol                          0
total_lake_area                  0
distance_to_lines_km             0
max_voltage                      0
total_energy_consumption_mwh    19
data_centers_count               0
pct_military                     0
pct_protected                    0
dtype: int64

In [73]:
X = X.dropna()
y = y.loc[X.index]

In [74]:
from sklearn.model_selection import train_test_split

#split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#create model
logit = LogisticRegression(class_weight='balanced', max_iter=1000)

logit.fit(X_train_scaled, y_train)

#evaluate
from sklearn.metrics import classification_report

y_pred = logit.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      0.78      0.88       421
           1       0.06      0.75      0.11         8

    accuracy                           0.78       429
   macro avg       0.53      0.77      0.50       429
weighted avg       0.98      0.78      0.86       429



#### Logistic Regression Coefficients (Log-Odds Effects)

In [78]:
coef = pd.Series(logit.coef_[0], index=features)
coef


pga_max                         0.514731
pct_sfha                        0.201498
population_density             -0.437354
dist_to_lakes_km               -0.401219
avg_vol                        -0.159358
total_lake_area                 0.503831
distance_to_lines_km           -0.615022
max_voltage                     1.110668
total_energy_consumption_mwh    0.895813
data_centers_count             -1.642290
pct_military                   -0.141479
pct_protected                   0.041105
dtype: float64

#### Relative Feature Importance Based on Logistic Regression Coefficients
ranking features by strength of effect (ignoring direction)

In [79]:
coef.abs().sort_values(ascending=False)

data_centers_count              1.642290
max_voltage                     1.110668
total_energy_consumption_mwh    0.895813
distance_to_lines_km            0.615022
pga_max                         0.514731
total_lake_area                 0.503831
population_density              0.437354
dist_to_lakes_km                0.401219
pct_sfha                        0.201498
avg_vol                         0.159358
pct_military                    0.141479
pct_protected                   0.041105
dtype: float64

#### Turn log-odds coefficient into relative importance shares
The original result from running the logistic regression model represents the change in log-odds of a county having a nuclear plant for a one standard deviation increase in each feature, which is not directly comparable to the weights previously identified.

After taking the absolute values of the coefficients and normalizing them so that they sum to 1, we convert them into relative importance shares, which can then be compared to the weights identified earlier.

In [82]:
criteria_weights = {
 'pga_max': 0.25860088985088986,
 'pct_sfha': 0.17526755651755652,
 'population_density': 0.13360088985088983,
 'dist_to_lakes_km': 0.10582311207311207,
 'avg_vol': 0.08498977873977874,
 'total_lake_area': 0.06832311207311208,
 'distance_to_lines_km': 0.05443422318422319,
 'max_voltage': 0.04252946127946128,
 'total_energy_consumption_mwh': 0.032112794612794614,
 'data_centers_count': 0.02285353535353535,
 'pct_military': 0.014520202020202022,
 'pct_protected': 0.006944444444444444
}

coef_abs = coef.abs()
coef_norm = coef_abs / coef_abs.sum()

common = sorted(list(set(features) & set(criteria_weights.keys())))

comparison = pd.DataFrame({
    'MCDA_weight': pd.Series(criteria_weights).loc[common],
    'Model_importance': coef_norm.loc[common]
})

comparison = comparison.sort_values(by='MCDA_weight', ascending=False)
comparison

,MCDA_weight,Model_importance
pga_max,0.258601,0.077236
pct_sfha,0.175268,0.030235
population_density,0.133601,0.065626
dist_to_lakes_km,0.105823,0.060204
avg_vol,0.084990,0.023912
total_lake_area,0.068323,0.075601
distance_to_lines_km,0.054434,0.092285
max_voltage,0.042529,0.166658
total_energy_consumption_mwh,0.032113,0.134418
data_centers_count,0.022854,0.246428


#### Summary
**Key Findings: MCDA vs Model Importance**  

Risk factors: pga_max and pct_sfha are highly weighted in MCDA but have lower importance in the model.  
Infrastructure: max_voltage and distance_to_lines_km are more important in the model than in MCDA.  
Demand signals stronger than expected: total_energy_consumption_mwh and especially data_centers_count show high model importance.  
Water variables mixed: total_lake_area aligns somewhat, while avg_vol is less important than expected.  
Agreement on low-impact factors: pct_protected and pct_military remain low in both approaches.